# Fitting NS1 from raw waveform: causal mel, SincNet, ICNet

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/urancon/deepSTRF/blob/develop/examples/fit_ns1_linear_from_waveform.ipynb)

deepSTRF accepts raw audio waveforms in addition to precomputed
spectrograms. This notebook walks through the three shipped
`wav2spec` front-ends on NS1:

1. **`CausalMelSpectrogram`** — non-learnable causal log-mel. The
   pipeline-validation baseline.
2. **`SincNet`** — parametric bandpass filterbank (Ravanelli & Bengio
   2018). The learnable spectrogram.
3. **`ICNetFrontend`** + **`ICNet`** — full encoder + decoder model
   from Drakopoulos et al. (Nat. Mach. Intell. 2025), ported to deepSTRF
   with auto-adapted strides for NS1's 16 kHz / 5 ms binning.

The data side is just `NS1Dataset(return_waveform=True, audio_fs=16000)` —
the dataset returns `(1, T_audio=79920)` mono float tensors aligned to
the existing 999-bin neural response grid. See
[`wav2spec.md`](../docs/_source/md/wav2spec.md) for the slot contract.


## Setup — Google Colab

If you're running on Google Colab, install deepSTRF from source. On a
local install (`pip install -e .`) the cell is a no-op.

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/urancon/deepSTRF.git
    print('deepSTRF installed from GitHub.')
else:
    print('Local environment — assuming deepSTRF is already importable.')

## Imports

In [ ]:
%matplotlib inline
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset

from deepSTRF.datasets.audio.ns1_drc import NS1Dataset
from deepSTRF.models.audio import Linear, ICNet
from deepSTRF.models.wav2spec import CausalMelSpectrogram, SincNet
from deepSTRF.training import Fitter
from deepSTRF.utils import neural_collate, compare_wav2spec_to_groundtruth

## 1. Load NS1 in waveform mode

We instantiate the dataset twice — once in waveform mode (for the
wav-input models) and once in default spectrogram mode (as the
ground-truth spec for visual comparison + the spec-side baseline
training arm). Responses are bit-identical between the two; only the
`self.stims` representation differs.

In [ ]:
ds_wav = NS1Dataset(return_waveform=True, audio_fs=16000, download=True)
ds_spec = NS1Dataset(download=True)

N = ds_wav.N_neurons
T_neural = ds_wav.stims[0].shape[-1] // 80   # 999 at 16 kHz / 5 ms
print(f'NS1: N={N} cells | T_audio={ds_wav.stims[0].shape[-1]} ({ds_wav.audio_fs} Hz × 4.995s) | T_neural={T_neural} (5 ms bins)')
print(f'wav stim 0 shape: {tuple(ds_wav.stims[0].shape)}')
print(f'spec stim 0 shape: {tuple(ds_spec.stims[0].shape)}')

## 2. Visual sanity: causal mel vs ground-truth spec

`compare_wav2spec_to_groundtruth` returns the wav2spec output, the
ground-truth precomputed spec, and a 3-panel figure (pred | truth |
difference, all z-scored). The causal mel doesn't bit-match the
precomputed spec (different `n_fft` / window / mel-scale), but the
qualitative match is strong and the downstream Linear readout absorbs
the filterbank-shape mismatch.

In [ ]:
mel = CausalMelSpectrogram(audio_fs=16000, n_mels=34, hop_ms=5.0, win_ms=25.0,
                            f_min=300.0, f_max=8000.0)
for stim_idx in (0, 7, 12):
    pred, truth, fig = compare_wav2spec_to_groundtruth(
        ds_wav, mel, stim_idx=stim_idx,
        ground_truth_stims=ds_spec.stims,
        suptitle=f'CausalMelSpectrogram vs ground truth — NS1 stim {stim_idx}'
    )
    r = np.corrcoef(pred.ravel(), truth.ravel())[0, 1]
    print(f'stim {stim_idx}: pred-vs-truth r = {r:.3f}')
    plt.show()

## 3. A small fit-and-report helper

Same training recipe (AdamW lr=1e-3, 50 epochs, no early stopping) for
every arm, so the only thing changing across rows of the final table
is the front-end. The split is the same as in the
`strf_parameterizations_ns1` notebook (14 / 3 / 3 by stim index).

In [ ]:
def fit_and_report(ds, model, label, max_epochs=50, lr=1e-3):
    train = DataLoader(Subset(ds, list(range(14))),     batch_size=1, shuffle=True,  collate_fn=neural_collate)
    val   = DataLoader(Subset(ds, list(range(14, 17))), batch_size=1, shuffle=False, collate_fn=neural_collate)
    test  = DataLoader(Subset(ds, list(range(17, 20))), batch_size=1, shuffle=False, collate_fn=neural_collate)
    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    fitter = Fitter(model, train, val, optimizer=optim, device='cpu',
                    max_epochs=max_epochs, patience=max_epochs,
                    monitor='val_cc_norm', mode='max', log_fn=lambda d: None)
    t0 = time.time()
    history = fitter.fit()
    elapsed = time.time() - t0
    cc_norm = fitter.evaluate(test)['cc_norm'].cpu()
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return dict(label=label, cc_norm=cc_norm,
                mean=float(cc_norm.mean()), median=float(cc_norm.median()),
                n_params=n_params, elapsed=elapsed)

## 4. Three Linear arms: spec, wav+mel, wav+sincnet

Each arm uses the same `Linear(F=34, T_strf=9, N)` readout — only the
input front-end changes.

In [ ]:
results = []
F_bands, T_strf = 34, 9

# A: spec input baseline (default wav2spec=Identity)
torch.manual_seed(0)
m_a = Linear(n_frequency_bands=F_bands, temporal_window_size=T_strf, out_neurons=N)
results.append(fit_and_report(ds_spec, m_a, 'spec (baseline)'))

# B: wav input + causal mel
torch.manual_seed(0)
m_b = Linear(n_frequency_bands=F_bands, temporal_window_size=T_strf, out_neurons=N,
             wav2spec=CausalMelSpectrogram(audio_fs=16000, n_mels=F_bands,
                                            hop_ms=5.0, win_ms=25.0,
                                            f_min=300.0, f_max=8000.0))
results.append(fit_and_report(ds_wav, m_b, 'wav + causal mel'))

# C: wav input + SincNet (envelope mode, mel-init)
torch.manual_seed(0)
m_c = Linear(n_frequency_bands=F_bands, temporal_window_size=T_strf, out_neurons=N,
             wav2spec=SincNet(audio_fs=16000, n_filters=F_bands,
                              kernel_size=251, hop_ms=5.0,
                              f_min=300.0, f_max=8000.0,
                              init='mel', activation='logabs', envelope=True))
results.append(fit_and_report(ds_wav, m_c, 'wav + sincnet (env)'))

for r in results:
    print(f"  {r['label']:22s}  mean cc_norm = {r['mean']:.4f}  median = {r['median']:.4f}  "
          f"params = {r['n_params']:>7,d}  {r['elapsed']:.0f}s")

## 5. Bonus: SincNet learns interpretable cutoffs

After training, the `f1` / `f2` parameters drift from their mel-spaced
initial positions. Plotting them reveals which frequency bands the
model finds informative for NS1 cortex prediction.

In [ ]:
with torch.no_grad():
    f1 = m_c.wav2spec.f1.cpu().numpy()
    f2 = m_c.wav2spec.f2.cpu().numpy()
fig, ax = plt.subplots(figsize=(7, 3))
idx = np.arange(len(f1))
ax.fill_between(idx, f1, f2, alpha=0.4, label='learned passband')
ax.plot(idx, (f1 + f2) / 2, 'o-', ms=4, label='center freq')
ax.set_xlabel('filter index')
ax.set_ylabel('frequency (Hz)')
ax.set_title('SincNet learned bandpass cutoffs after fitting NS1')
ax.legend()
plt.tight_layout()
plt.show()

## 6. ICNet (Drakopoulos et al. 2025) on NS1

ICNet is a much deeper model (5.1 M params on NS1 vs Linear's ~37 k)
and was trained in the paper on **midbrain** (IC) data in **gerbils**,
not cortex (A1) in ferrets like NS1. Don't expect it to beat a
well-tuned cortex-specific Linear model out of the box. The point of
this section is to demonstrate that the ICNet architecture ports
cleanly into deepSTRF's slot machinery (auto-adapted strides for
NS1's `audio_fs · dt_ms / 1000 = 80` samples per bin).

In [ ]:
torch.manual_seed(0)
m_icnet = ICNet(audio_fs=16000, out_neurons=N, dt_ms=5.0)
print(f'ICNet on NS1: strides={m_icnet.wav2spec.encoder_strides}  '
      f'params={sum(p.numel() for p in m_icnet.parameters()):,}')

# WARNING: ICNet has 5.1M params; on CPU each epoch is slow (~minutes).
# Comment the next line out if you're on a low-resource machine — or
# better, move to GPU (model_icnet.to('cuda')) and bump max_epochs.
results.append(fit_and_report(ds_wav, m_icnet, 'wav + ICNet', max_epochs=30, lr=1e-3))

print()
print(f'{"front-end":<22s}  {"mean cc_norm":>12s}  {"median":>8s}  {"params":>9s}  {"time":>5s}')
for r in results:
    print(f'{r["label"]:<22s}  {r["mean"]:>12.4f}  {r["median"]:>8.4f}  {r["n_params"]:>9,d}  {r["elapsed"]:>4.0f}s')

## Takeaways

- **Wav-input + causal mel matches the precomputed-spec baseline.**
  Confirms the `wav2spec` slot mechanics work end-to-end.
- **SincNet (envelope mode) learns reasonable cutoffs but underperforms
  fixed mel** when paired with a thin Linear readout. The learnable
  flexibility is most useful when SincNet is part of a deeper stack —
  exactly the role it plays inside ICNet.
- **ICNet ports cleanly** to NS1 with stride retuning (`[2,2,2,2,5]`
  vs paper's `[2,2,2,2,2]`); the architecture is the same. Performance
  on cortex is a different question — see the paper's IC vs A1
  caveats.

See [`wav2spec.md`](../docs/_source/md/wav2spec.md) for the slot
contract and how to write your own front-end.